# Using `data.features` to generate a full dataframe for training

I tried to create a system which makes it easier to cookup recipes for modeling-ready dataframes. This is split into
* `features.py`: methods for accessing specific features (like crops, nitrate_rolling_avg, etc.) Add new feature methods to this file!
* `transforms.py`: methods for transforming date, performing merges, aggregations etc. that you don't want to rewrite every time you use them. Add your transformations!
* `recipes.py`: file for end-to-end `site_uid -> DataFrame` methods. Add your recipes!

The two recipes present right now are `isaac_df1()` and `preet_df1`. The latter is my attempt to replicate Preet's df build. This is a diagram of how `isaac_df1()` works:

<p align = center>
<img src = 'images/scribbles.png' width = 1500></img>
</p>

First, here are some super terse recipes.

In [ ]:
import sys
sys.path.insert(0, "../")

from src.features.features import (
    agg_crops,
    agg_surplus,
    agg_weather_w_lag,
    daily_nitrate,
    lagged_sensor_nitrate,
    nitrate_rolling,
    nitrate_avg_seasonal,
    nitrate_avg_calendar,
    doy_climatology_pure_signal,
)
from src.features.transformers import flatten_buckets, merge_on_date, match_seasonal


def _covariates(site, edges=[], new_name_of_target="nitrate_con"):
    """weather (travel-time lagged buckets) + crops/surplus (exp-decay buckets) + pure calendar."""
    wb = flatten_buckets(agg_weather_w_lag(site, edges=edges, water_velocity=new_name_of_target))
    cb = flatten_buckets(agg_crops(site, edges=edges, lam=10_000, exp=True))
    sb = flatten_buckets(agg_surplus(site, edges=edges, lam=10_000, exp=True))
    n_daily = daily_nitrate(site).rename(new_name_of_target)
    doy = doy_climatology_pure_signal(n_daily)  # doy_sin/doy_cos
    return n_daily, [wb, cb, sb, doy]


def recipe_A(site, edges=[], new_name_of_target="nitrate_con"):
    """Covariates only: weather + land-use + pure calendar. No nitrate-derived features."""
    n_daily, parts = _covariates(site)
    out = merge_on_date([n_daily, *parts], spine=n_daily.index)
    return out.dropna(subset=[new_name_of_target]).reset_index(drop=True)


def recipe_B(site):
    """A + the site's OWN past nitrate (autoregressive; sensor sees its own history)."""
    n_daily, parts = _covariates(site)
    own = [lagged_sensor_nitrate([site], shift=k) for k in (1, 2, 3, 7, 14, 30)]
    return merge_on_date([n_daily, *parts, *own], spine=n_daily.index)


def recipe_C(site):
    """A + cross-site climatology (causal) + past nitrate of all other sensors."""
    n_daily, parts = _covariates(site)
    dates = n_daily.index
    clim = [
        nitrate_rolling("7D", center=False).rename("nroll_7"),
        nitrate_rolling("30D", center=False).rename("nroll_30"),
        nitrate_avg_calendar("D").rename("ncal_d"),
        match_seasonal(dates, nitrate_avg_seasonal("D")).rename("ndoy"),
        match_seasonal(dates, nitrate_avg_seasonal("W")).rename("nwoy"),
        match_seasonal(dates, nitrate_avg_seasonal("M")).rename("nmoy"),
    ]
    neigh = [lagged_sensor_nitrate([site], shift=k) for k in (1, 3, 7)]
    return merge_on_date([n_daily, *parts, *clim, *neigh], spine=dates)

## Accessing recipes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../")

from src.data.access import get_data
from src.features.recipes import recipe_REG

uid = "WQS0039"

# the important bit: the current canned regression recipe (feature frame + target).
# bucket edges / travel-time lag / rolling windows are baked into the shipped recipe now
# (see recipe_REG in src/features/recipes.py); the manual cells below show how to roll your own.
df = recipe_REG(uid)
df.columns


## Simple recipe with 2 x buckets, no lag, no surplus data.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../")

from src.data.access import get_data
from src.features.features import agg_crops, agg_weather, daily_nitrate
from src.features.transformers import merge_on_date, flatten_buckets

def recipe(site_uid):
    # aggregate the weather grid into 2 buckets
    # node_id --> b0 if dist(node, sensor) < 50,000
    #         --> b1 if dist(node, sensor) > 50,000
    edges = [50_000]
    cb = agg_crops(site_uid=site_uid, edges=edges)
    wb = agg_weather(site_uid=site_uid, edges=edges)
    
    # flatten the buckets
    c_wide = flatten_buckets(cb)
    w_wide = flatten_buckets(wb)
    
    # get the target
    target = daily_nitrate(site_uid=site_uid) 
    
    # merge into one dataframe
    # the year of c_wide is broadcast to the daily date of w_wide and target
    return merge_on_date([c_wide, w_wide, target])

print(recipe("WQS0039").columns)

You can also aggregate the whole grid into one value with this bucket approach, use an empty edges list.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../")

from src.data.access import get_data
from src.features.features import agg_crops, agg_weather, daily_nitrate
from src.features.transformers import merge_on_date, flatten_buckets

def recipe(site_uid):
    # aggregate the weather grid into 2 buckets
    # node_id --> b0 if dist(node, sensor) < 50,000
    #         --> b1 if dist(node, sensor) > 50,000
    edges = [400000]
    cb = agg_crops(site_uid=site_uid, edges=edges)
    wb = agg_weather(site_uid=site_uid, edges=edges)
    
    # flatten the buckets
    c_wide = flatten_buckets(cb)
    w_wide = flatten_buckets(wb)
    
    # get the target
    target = daily_nitrate(site_uid=site_uid) 
    
    # merge into one dataframe
    # the year of c_wide is broadcast to the daily date of w_wide and target
    return merge_on_date([c_wide, w_wide, target])

print(recipe("WQS0039").columns)

## Adding in exponential decay

The methods `agg_crops_by_bucket`, `agg_weather_by_bucket` etc are wrappers for the transformation method `agg_by_bucket` with sensible defaults. The aggregation methods are given in `transformers._sensible_agg_dicts`. These only take averages or sums weighted by area of cell in basin, they do NOT perform any kind of distance weighting. The idea is that the distance from node should be captured by the buckets themselves, hopefully the model can learn that crop data from bucket 2 is less important than crop data from bucket 0.

Distance weighting can be combined with this bucket aggregation. Here's the fast way to do it:

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../")

from src.data.access import get_data
from src.features.features import agg_crops, agg_weather, daily_nitrate
from src.features.transformers import merge_on_date, flatten_buckets, _bucket_map, _agg_dicts

def recipe(site_uid, lam, edges):
    # aggregate the weather grid into 2 buckets with exponential decay
    # NOTE THE EXP = TRUE
    # node_id --> b0 if dist(node, sensor) < 50,000
    #         --> b1 if dist(node, sensor) > 50,000
    cb = agg_crops(site_uid=site_uid, edges=edges, lam=lam, exp=True, normalize=True)
    wb = agg_weather(site_uid, edges)
    
    # flatten the buckets
    c_wide = flatten_buckets(cb)
    w_wide = flatten_buckets(wb)
    
    # get the target
    target = daily_nitrate(site_uid=site_uid) 
    
    # merge into one dataframe
    # the year of c_wide is broadcast to the daily date of w_wide and target
    return merge_on_date([c_wide, w_wide, target])

site = "WQS0115"
lam = 20_000
edges = [] # one big bucket, i.e. aggregate over whole basin

d = get_data(site)
frac = d.grid.frac_cell_in_basin.values
exp = np.exp(-d.grid.dist_to_sensor.values / lam)

corn = d.crops[d.crops["year"] == 2023].Corn.values
agg_corn = np.average(corn, weights=exp*frac)

cb = recipe(site, lam, edges)
print(f"Manual exp weighted corn in 2023: {agg_corn}")
print(f"recipe exp weighted corn in 2023: {cb[cb.date == "2023-01-01"].Corn.values[0]}")

And here's the manual way to do it (both should be equivalent to Preet's method up to scaling)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../")

from src.data.access import get_data
from src.features.features import daily_nitrate
from src.features.transformers import agg_grid_to_buckets, merge_on_date, flatten_buckets, _bucket_map, _agg_dicts

# this creates an aggregation function tailored to the site
def _exp_curry(site_uid, lam, avg_like = False):
    grid = get_data(site_uid=site_uid).grid
    fracs = grid.frac_cell_in_basin.values
    dist_exp =  np.exp(-grid.dist_to_sensor.values/lam)
    weights = pd.Series(fracs*dist_exp, index=grid.node_id)
    def _exp_decay_weighting(values):
        #print(f"weight len: {len(weights[values.index])}")
        #print(f"values len: {len(values)}")
        if avg_like:
            return float(np.average(values, weights=weights.loc[values.index]))
        else:
            return float(np.dot(values, weights.loc[values.index]))
    
    return _exp_decay_weighting

# here is a custom wrapper for agg_grid_to_buckets
# specialized to our needs
def exp_agg_to_bucket(site_uid, lam, edges, avg_like=True):
    # maps node_id to buckets
    mapping = _bucket_map(site_uid, edges)
    
    # column names to aggregation dict
    # you can also write you own but this
    # matches column names of crops, surplus, weather
    # automatically
    c_dict, s_dict, w_dict = _agg_dicts(land_use_func=_exp_curry(site_uid, lam, avg_like=True), weather_func="mean")
    
    d = get_data(site_uid)
    crops, surplus, weather = d.crops, d.surplus, d.weather
    
    cb = agg_grid_to_buckets(crops, mapping, keys=["year"], col_agg=c_dict)
    sb = agg_grid_to_buckets(surplus, mapping, keys=["year"], col_agg=s_dict)
    wb = agg_grid_to_buckets(weather, mapping, keys=["date"], col_agg=w_dict)
    return cb, sb, wb

def recipe(site_uid, lam, edges):
    # aggregate the weather grid into 2 buckets with exponential decay
    # node_id --> b0 if dist(node, sensor) < 50,000
    #         --> b1 if dist(node, sensor) > 50,000
    cb, _, wb = exp_agg_to_bucket(site_uid=site_uid, edges=edges, lam=lam)
    
    # flatten the buckets
    c_wide = flatten_buckets(cb)
    w_wide = flatten_buckets(wb)
    
    # get the target
    target = daily_nitrate(site_uid=site_uid) 
    
    # merge into one dataframe
    # the year of c_wide is broadcast to the daily date of w_wide and target
    return merge_on_date([c_wide, w_wide, target])

site = "WQS0115"
lam = 20_000
edges = [] # one big bucket, i.e. aggregate over whole basin

d = get_data(site)
frac = d.grid.frac_cell_in_basin.values
exp = np.exp(-d.grid.dist_to_sensor.values / lam)

corn = d.crops[d.crops["year"] == 2023].Corn.values
agg_corn = np.average(corn, weights=exp*frac)

cb = recipe(site, lam, edges)
print(f"Manual exp weighted corn in 2020: {agg_corn}")
print(f"recipe exp weighted corn in 2020: {cb[cb.date == "2023-01-01"].Corn.values[0]}")

## Recipe for a bunch of lags and lots of rolling

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../")

from src.data.access import get_data
from src.features.features import daily_nitrate, agg_crops, agg_surplus, agg_weather, nitrate_rolling, nitrate_avg_calendar, nitrate_avg_seasonal, doy_climatology_pure_signal
from src.features.transformers import agg_grid_to_buckets, merge_on_date, flatten_buckets, _bucket_map, _agg_dicts, bucket_lags, lag_buckets, match_seasonal

VEL = 0.8 # m/s
EDGES = [50_000, 100_000, 150_000, 200_000] # 5 buckets
LAM = 10000

def recipe(site_uid):
    cb = agg_crops(site_uid, edges=EDGES, lam=LAM, exp=True)
    sb = agg_surplus(site_uid, edges=EDGES, lam=LAM, exp=True)
    wb = agg_weather(site_uid, edges=EDGES)
    
    # first we lag by bucket
    lag = bucket_lags(site_uid=site_uid, water_velocity=VEL, edges=EDGES)
    print("The calculated per-bucket lag:", lag)
    wb_lag = lag_buckets(wb, lags=lag, date_col="date", bucket_col="bucket")
    
    # next let's just add in a bunch of copies of the weather
    # all lagged at different numbers of days
    # notice the empty edges parameter
    wb1 = agg_weather(site_uid, edges=[])
    val_cols = ['precip_in_1d', 'max_temp',
       'min_temp', 'max_rel_humidity', 'min_rel_humidity', 'vpd', 'solar_rad',
       'evapotranspiration', 'fuel_moisture_1000h']
    wb1 = wb1.rename({col : col + "_lag1" for col in val_cols})
    wb2 = wb1.rename({col : col + "_lag2" for col in val_cols})
    wb3 = wb1.rename({col : col + "_lag3" for col in val_cols})
    
    wb1 = lag_buckets(wb1, lags=[1], date_col="date")
    wb2 = lag_buckets(wb2, lags=[2], date_col="date")
    wb3 = lag_buckets(wb3, lags=[3], date_col="date")
    
    # target: daily nitrate 
    n_daily = daily_nitrate(site_uid=site_uid).rename("nitrate_con")
    dates = n_daily.index
    
    # now we'll add in a bunch of random nitrate data
    # cross-site date-keyed reference features (distinct names so they don't collide)
    n_rolling_3D = nitrate_rolling(window="3D", center=False).rename("nitrate_roll")
    n_cal_D = nitrate_avg_calendar(freq="D").rename("nitrate_cal_d")
    n_cal_W = nitrate_avg_calendar(freq="W").rename("nitrate_cal_w")
    n_cal_M = nitrate_avg_calendar(freq="M").rename("nitrate_cal_m")
    pure_signal = doy_climatology_pure_signal(n_daily)  # date-indexed (doy_sin/doy_cos)

    # seasonal nitrate averages mapped onto the calendar dates
    def _help(d, name):
        return match_seasonal(dates=dates, seasonal=d).rename(name)

    n_doy = _help(nitrate_avg_seasonal(freq="D"), "nitrate_doy")
    n_woy = _help(nitrate_avg_seasonal(freq="W"), "nitrate_woy")
    n_moy = _help(nitrate_avg_seasonal(freq="M"), "nitrate_moy")
   
    # flatten the buckets 
    w_wide = flatten_buckets(wb_lag)
    c_wide = flatten_buckets(cb)
    s_wide = flatten_buckets(sb)
    
    # and merge everything
    out = merge_on_date(
        [n_daily, n_rolling_3D, n_cal_D, n_cal_W, n_cal_M, pure_signal, n_doy, n_woy, n_moy, w_wide, c_wide, s_wide, wb1, wb2, wb3],
        spine=dates,
    )
    
    return out
   
import math 
site = "WQS0115"
df = recipe(site)